<a href="https://colab.research.google.com/github/lawho13/Influence-Parameter-Calibration-of-UBI-Equilibrium/blob/main/risk_aversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Risk Aversion Estimate
We replicated Szpiro's paper on using insurance claims data and wealth data to empirically estimate an economy's coefficient of risk aversion.

In the paper, Szpiro assumes a risk aversion function
$$
r(W) = \frac{c}{W^h},
$$
where $c$ is the empirically estimated coefficient of relative risk aversion and $W^h$ is wealth raised to the power $h$.

The value of $h$ determines how risk aversion changes with wealth. Values of $h < 1$ indicate decreasing relative risk aversion (DRRA), while values of $h > 1$ imply increasing relative risk aversion (IRRA). When $h = 1$, relative risk aversion is constant (CRRA), and the function simplifies to
$$
r(W) = \frac{c}{W}.
$$

In our replication, we don't fix h to be 1 leading to some unrealistic estimates for relative risk aversion in some economies.

# Methodology

Szpiro models the utility maximizing number of assets insured as,
$$
I = W - \frac{\lambda}{r(W)}
$$
and the claims rate as $$Q = Iq$$
Rearranging, you get that $$Q = qW-\frac{q\lambda}{r(W)}$$

Plugging in the risk aversion function $r(W) = \frac{c}{W^h}$:

$$
Q_t = qW_t - \frac{q}{c} \lambda_t W_t^h.
$$

We estimate the following regression:
$$
Q_t = b W_t + n (\lambda_t W_t^h),
$$
where
$$
b = q, \quad n = -\frac{q}{c}.
$$

Rearranging and solving for $c$, we get:
$$
n = -\frac{b}{c} \;\;\Rightarrow\;\; c = -\frac{b}{n}.
$$

To estimate $h$, Szpiro performs a grid search over $h \in [-5, 5]$, fitting OLS models for each value. The optimal $h$ minimizes the sum of squared errors (SSE), and the corresponding regression coefficients provide estimates for $b$ and $n$.

Important things to note: All data is computed per capita of economy in constant USD, the regression target Q and computation of $\lambda$ is to be done with 5 year moving averages of Q to smooth the noise in Insurance claims data.

# Writeup and Notes
In our early stages of fitting, OECD net wealth data was extremely incomplete. This led to poor and unreasonable estimates. In an extension of his paper (that wasn't discovered until later on in our research process), Szpiro mentions using GNP as a proxy for the incomplete wealth data (Paper [Szpiro: Relative Risk Aversion Around The World](https://www.sciencedirect.com/science/article/pii/0165176586900728)) which produced better results. For economies with less than 5 missing entries for a feature, we did linear interpolation and with mroe than 5 missing entries we dropped the economy. In the end, our Q was estimated using OECD compiled Gross Claims Payment data, W using GNI (GNP) data from 2009-2024. Some economies found the optimal ${h<0}$ leading to a negative ${c}$

After finishing, the results were written into a csv for all economies with sufficient data. CSV is indexed by 'REF_AREA', with col for Year, optimal_h, rra ($RRA = W_t(\frac{c}{W_t^h}))$ , ara ($ARA = \frac{c}{W_t^h}$)

In [ ]:
import statsmodels.api as sm
from google.colab import drive, os
drive.mount('/content/drive', force_remount=True)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

Mounted at /content/drive


In [ ]:
gross_claims_premiums = pd.read_csv("/content/drive/My Drive/490 project/gross_claim_premium.csv")
premium_capita = pd.read_csv("/content/drive/My Drive/490 project/premium_density.csv")
gnp = pd.read_csv('/content/drive/My Drive/490 project/GNP.csv')
conversion_rates = pd.read_csv("/content/drive/My Drive/490 project/conversion_rates.csv")

# Load and format data

In [ ]:
# @title
# 1. Define the identifying columns that should not be melted
id_vars = ['Series Name', 'Series Code', 'Country Name', 'Country Code']

# 2. Melt the dataframe
# This turns the 'Year' columns into rows
df_long = gnp.melt(
    id_vars=id_vars,
    var_name='TIME_PERIOD',
    value_name='OBS_VALUE'
)

# 3. Clean the 'TIME_PERIOD' column
# (Removes the "[YR2009]" suffix to leave just "2009")
df_long['TIME_PERIOD'] = df_long['TIME_PERIOD'].str.extract('(\d{4})')

# 4. Clean the 'OBS_VALUE' column
# Converts strings to numbers and turns ".." into NaN
df_long['OBS_VALUE'] = pd.to_numeric(df_long['OBS_VALUE'], errors='coerce')

# 5. Select only the requested columns
df_final = df_long[['TIME_PERIOD', 'Country Name', 'Country Code', 'OBS_VALUE']]
df_final = df_final.sort_values(['Country Name', 'TIME_PERIOD'])

<>:15: SyntaxWarning: invalid escape sequence '\d'
<>:15: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_553/1422822298.py:15: SyntaxWarning: invalid escape sequence '\d'
  df_long['TIME_PERIOD'] = df_long['TIME_PERIOD'].str.extract('(\d{4})')


In [ ]:
# wealth data is in national currency and current prices, based on chosen economies will need to convert to 2022 USD.
cpi = pd.read_csv('/content/drive/My Drive/490 project/us_cpi.csv')
cpi.head()
cpi['year'] = pd.to_datetime(cpi['observation_date']).dt.year
cpi_2022 = cpi.loc[cpi['year'] == 2022, 'CPIAUCSL'].values[0]

cpi['adjustment'] = cpi_2022 / cpi['CPIAUCSL']
adjustment_map = cpi.set_index('year')['adjustment'].to_dict()

In [ ]:
# clean conversion rates
def clean_conversion_rates(df):
  df = df.copy()
  df = df.rename(columns=lambda x: re.search(r'\d+', x).group() if re.search(r'\d+', x) else x)
  df['Country Name'] = df['Country Name'].replace('Korea, Rep.', 'Korea')
  df['Country Name'] = df['Country Name'].replace('Russian Federation', 'Russia')
  df['Country Name'] = df['Country Name'].replace('Turkiye', 'Türkiye')

  return df

conversion_rates = clean_conversion_rates(conversion_rates)

In [ ]:
def conversion_rate(location, year):
  '''
  local currency to USD by using annual average compiled by world bank
  '''
  year_str = str(year)

  # filter for the specific country
  country_row = conversion_rates[conversion_rates['Country Name'] == location]

  # check if the country exists and if the year column exists for that country
  if not country_row.empty and year_str in country_row.columns:
    rate_value = country_row[year_str].values[0]
    if pd.isna(rate_value) or str(rate_value).strip() == '..':
      return np.nan
    else:
      return float(rate_value)
  else:
    return np.nan

def build_n_table(df = gross_claims_premiums):
  gross = df[(df['Measure'] == 'Gross written premiums') &
    (df['Insurance business']=='Total') &
    # (df['Contract type']=='Total') &
    (df['Ownership']=='All undertakings') &
    (df['Confidentiality status']=='Free (free for publication)') &
    (df['Observation status']=='Normal value') &
    (df['Risk location']=='Not applicable')][['OBS_VALUE', 'TIME_PERIOD', 'Reference area']]

  capita = premium_capita[['OBS_VALUE', 'TIME_PERIOD', 'Reference area']]

  # rescale
  gross['OBS_VALUE'] *= 10**6

  # rename
  gross = gross.rename(columns={'OBS_VALUE': 'gross_premium'})
  capita = capita.rename(columns={'OBS_VALUE': 'capita_premium'})

  # gross = (
  #   gross.groupby(['TIME_PERIOD', 'Reference area'], as_index=False)
  #   ['gross_premium']
  # )

  capita = capita.rename(columns={'OBS_VALUE': 'capita_premium'})

  merged = pd.merge(gross, capita, on=['TIME_PERIOD', 'Reference area'])
  merged['n'] = merged['gross_premium'] / merged['capita_premium']

  return merged[['TIME_PERIOD', 'Reference area', 'n']]

def clean_gnp(df, n_table):
  df = df.copy()

  # Convert TIME_PERIOD to integer to match n_table
  df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce').astype(int)

  # Rename columns to match the merge key in n_table and other dataframes
  df = df.rename(columns = {'Country Name': 'Reference area',
                            'Country Code': 'REF_AREA'})

  # Calculate OBS_VALUE_converted
  df['OBS_VALUE_converted'] = df['OBS_VALUE'] * df['TIME_PERIOD'].map(adjustment_map)

  # Merge with n_table
  df = df.merge(n_table, on=['TIME_PERIOD', 'Reference area'], how='left')

  # Calculate gni_per_capita using the converted OBS_VALUE
  df['gni_per_capita'] = df['OBS_VALUE_converted'] / df['n'] # per capita computation

  df = df[df['TIME_PERIOD']!=2025]
  df = df[['TIME_PERIOD', 'Reference area', 'REF_AREA', 'gni_per_capita']]

  return df

def clean_gross_claims_premiums(df, n_table):
  df = df.copy()

  df = df[(df['Measure'] == 'Gross claims payments') &
    (df['Insurance business']=='Total') &
    (df['Ownership']=='All undertakings') &
    (df['Confidentiality status']=='Free (free for publication)') &
    (df['Observation status']=='Normal value') &
    (df['Risk location']=='Not applicable')]

  df['OBS_VALUE'] = df['OBS_VALUE']*10**6 # rescale for per capita computation

  df = df.merge(n_table, on=['TIME_PERIOD', 'Reference area'], how='left')
  df['claim_per_capita'] = df['OBS_VALUE'] / df['n'] # per capita computation

  df = df[['TIME_PERIOD', 'Reference area', 'REF_AREA', 'claim_per_capita']]

  # df = df.groupby(['TIME_PERIOD', 'Reference area', 'REF_AREA'], as_index=False)
  return df

def clean_premium_capita(df):
  df = df.copy()

  df['OBS_VALUE_converted'] = df['OBS_VALUE'] * df['TIME_PERIOD'].map(adjustment_map)

  df = df.rename(columns={'OBS_VALUE_converted': 'premium_per_capita'}) # in 2022 USD
  df = df[['TIME_PERIOD', 'Reference area', 'REF_AREA', 'premium_per_capita']]
  return df

In [ ]:
# d = gross_claims_premiums[(gross_claims_premiums['Measure'] == 'Gross claims payments') &
#     (gross_claims_premiums['Insurance business']=='Total') &
#     # (df['Contract type']=='Total') &
#     (gross_claims_premiums['Ownership']=='All undertakings') &
#     (gross_claims_premiums['Confidentiality status']=='Free (free for publication)') &
#     (gross_claims_premiums['Observation status']=='Normal value') &
#     (gross_claims_premiums['Risk location']=='Not applicable')]

In [ ]:
# d[d['REF_AREA']=='USA'][['TIME_PERIOD', 'REF_AREA', 'OBS_VALUE']]

In [ ]:
n_table = build_n_table()

cleaned_data = {
    'claims': clean_gross_claims_premiums(gross_claims_premiums, n_table), # claim payments per capita from 2009-2024
    'premium': clean_premium_capita(premium_capita), # gross written premiums in USD per capita 2009-2024
    'gnp_per_capita' : clean_gnp(df_final, n_table)
    # 'median_household_net_wealth': clean_median_household_net_wealth(median_household_net_wealth) # median household net wealth
}

final_df = (
    cleaned_data['claims']
    .merge(cleaned_data['premium'], on=['TIME_PERIOD', 'Reference area', 'REF_AREA'], how='left')
    .merge(cleaned_data['gnp_per_capita'], on=['TIME_PERIOD', 'Reference area', 'REF_AREA'], how='left')
)

In [ ]:
cleaned_data

{'claims':      TIME_PERIOD Reference area REF_AREA  claim_per_capita
 0           2012      Argentina      ARG               NaN
 1           2014         Brazil      BRA               NaN
 2           2023        Croatia      HRV               NaN
 3           2024        Croatia      HRV               NaN
 4           2017           Cuba      CUB               NaN
 ..           ...            ...      ...               ...
 789         2013          Spain      ESP        544.376546
 790         2014          Spain      ESP        535.910632
 791         2015          Spain      ESP        455.483819
 792         2016          Spain      ESP        450.780611
 793         2017          Spain      ESP        502.669423
 
 [794 rows x 4 columns],
 'premium':       TIME_PERIOD Reference area REF_AREA  premium_per_capita
 0            2009     Costa Rica      CRI                 NaN
 1            2009        Denmark      DNK                 NaN
 2            2010        Denmark      DNK 

In [ ]:
final_df.isna().sum()

,0
TIME_PERIOD,0
Reference area,0
REF_AREA,0
claim_per_capita,81
premium_per_capita,27
gni_per_capita,113


# Clean data (NAN)

In [ ]:
def clean_years(df = final_df):
  years = pd.DataFrame({'TIME_PERIOD': range(2009, 2025)})

  countries = pd.DataFrame({
      'Reference area': df['Reference area'].unique()
  })

  full_index = years.merge(countries, how='cross')

  df_full = full_index.merge(df, on=['TIME_PERIOD', 'Reference area'], how='left')
  df_full['REF_AREA'] = df_full.groupby('Reference area')['REF_AREA'].transform('first')

  return df_full

In [ ]:
# @title
def interpolate(df):
  '''
  Interpolate every missing value (may be concerning ways to fill but targets
  less concerning because will smooth with Moving Average anyways)

  Old Notebook investigation should be ok
  '''
  df = df.copy()

  df = df.sort_values(['REF_AREA', 'TIME_PERIOD'])

  df['claim_per_capita'] = (
    df.groupby('REF_AREA')['claim_per_capita']
    .transform(lambda x: x.interpolate(limit_direction='both'))
  )
  df['premium_per_capita'] = (
    df.groupby('REF_AREA')['premium_per_capita']
    .transform(lambda x: x.interpolate(limit_direction='both'))
  )

  df['gni_per_capita'] = (
      df.groupby('REF_AREA')['gni_per_capita']
      .transform(lambda x: x.interpolate(limit_direction='both'))
  )

  return df

In [ ]:
final_df = clean_years()

In [ ]:
final_df.groupby('Reference area')[[
    'claim_per_capita',
    'premium_per_capita',
    'gni_per_capita'
]].apply(lambda x: x.isna().sum())

,claim_per_capita,premium_per_capita,gni_per_capita
Reference area,,,
Argentina,4,3,3
Australia,0,0,0
Austria,6,3,6
Belgium,3,3,3
Bermuda,16,16,16
...,...,...,...
Tunisia,14,14,14
Türkiye,1,1,16
United Kingdom,0,0,0


In [ ]:
def drop_missing(df):
  df = df.copy()

  na_counts = df.groupby('Reference area')[[
      'claim_per_capita',
      'premium_per_capita',
      'gni_per_capita'
  ]].apply(lambda x: x.isna().sum())

  to_drop = na_counts[(na_counts > 5).any(axis=1)].index

  df = df[~df['Reference area'].isin(to_drop)]

  return df

In [ ]:
final_df = drop_missing(final_df)
final_df = interpolate(final_df)

# Final Check
final_df.groupby('Reference area')[[
    'claim_per_capita',
    'premium_per_capita',
    'gni_per_capita'
]].apply(lambda x: x.isna().sum())

,claim_per_capita,premium_per_capita,gni_per_capita
Reference area,,,
Argentina,0,0,0
Australia,0,0,0
Belgium,0,0,0
Chile,0,0,0
Colombia,0,0,0
Costa Rica,0,0,0
Czechia,0,0,0
Denmark,0,0,0
Estonia,0,0,0


In [ ]:
# final_df[final_df['REF_AREA']=='SWE']

In [ ]:
final_df.isna().sum()

,0
TIME_PERIOD,0
Reference area,0
REF_AREA,0
claim_per_capita,0
premium_per_capita,0
gni_per_capita,0


In [ ]:
# final_df[final_df['REF_AREA']=='SWE']

In [ ]:
final_df.shape

(592, 6)

# Visualization

In [ ]:
# @title
from ipywidgets import interact, Dropdown
import matplotlib.pyplot as plt
import seaborn as sns
def plot_interactive(df):
    ref_areas = sorted(df['REF_AREA'].dropna().unique())
    y_options = [
        'claim_per_capita',
        'premium_per_capita',
        'gni_per_capita'
    ]

    def plot(ref_area, y_col1, y_col2, y_col3):
        subset = df[df['REF_AREA'] == ref_area].sort_values('TIME_PERIOD')

        plt.figure(figsize=(12, 6))

        y_cols = [y_col1, y_col2, y_col3]
        colors = ['blue', 'orange', 'green']

        for y_col, color in zip(y_cols, colors):
            # Plot data
            sns.scatterplot(data=subset, x='TIME_PERIOD', y=y_col, label=y_col, color=color)
            sns.lineplot(data=subset, x='TIME_PERIOD', y=y_col, alpha=0.5, color=color)

            # Highlight NaNs
            nan_points = subset[subset[y_col].isna()]
            if not nan_points.empty:
                plt.scatter(
                    nan_points['TIME_PERIOD'],
                    [subset[y_col].min()] * len(nan_points),
                    marker='x',
                    color=color,
                    label=f'{y_col} NaN'
                )

        plt.title(f"{y_col1}, {y_col2}, and {y_col3} over time for {ref_area}")
        plt.xlabel("Year")
        plt.ylabel("Value")
        plt.legend()
        plt.show()

    interact(
        plot,
        ref_area=Dropdown(options=ref_areas, description='Country'),
        y_col1=Dropdown(options=y_options, description='Y Column 1'),
        y_col2=Dropdown(options=y_options, description='Y Column 2'),
        y_col3=Dropdown(options=y_options, description='Y Column 3')
    )

plot_interactive(final_df)

interactive(children=(Dropdown(description='Country', options=('ARG', 'AUS', 'BEL', 'CHE', 'CHL', 'COL', 'CRI'…

# Fitting

In [ ]:
def get_x_y(df, location, target):
    df = df.copy()

    # df = df[df['REF_AREA'] == location]
    df = df[df['Reference area'] == location]
    df = df.sort_values('TIME_PERIOD')

    # wealth
    W = df['gni_per_capita']
    P = df['premium_per_capita'].rolling(window=5).mean()
    Q = df['claim_per_capita'].rolling(window=5).mean()

    l = (P - Q) / Q # Difference from first attempt and second attempt was using the smoothed version of P, Q
    if target == 'claims':
        y = Q
    elif target == 'premiums':
        y = P
    else:
        raise ValueError("target must be 'claims' or 'premiums'")

    data = pd.DataFrame({
        'W': W, # Wealth
        'y': y, # Target
        'l': l # Loading
    }).dropna()

    return data['W'], data['y'], data['l']

def optimal_h(location, target):
    h_grid = np.linspace(-5, 5, 1001)

    W, y, loading = get_x_y(final_df, location, target)

    best_h = None
    best_res = None
    best_sse = np.inf

    W = np.array(W)
    y = np.array(y)
    loading = np.array(loading)

    for h in h_grid:
        Wh = (W ** h) * loading

        X = np.column_stack([W, Wh])
        res = sm.OLS(y, X).fit()

        sse = res.ssr

        if sse < best_sse:
            best_sse = sse
            best_h = h
            best_res = res


    h1_Wh = (W ** 1) * loading
    X_h1 = np.column_stack([W, h1_Wh])
    res_h1 = sm.OLS(y, X_h1).fit()

    sse_h1 = res_h1.ssr
    # Can run hypothesis test on SSE to determine if h=1 null hypothesis is rejected
    # haven't explored, for now, will just take 'optimal h' and compute
    return {
        "best_h": best_h,
        "best_model": best_res,
        "best_sse": best_sse,
        "sse_h1": sse_h1
    }



In [ ]:
def compute_risk(df, location, target):

    W, y, loading = get_x_y(df, location, target)

    h_results = optimal_h(location, target)
    h = h_results["best_h"]
    Wh = loading * (W ** h)
    X = np.column_stack([W, Wh])

    model = sm.OLS(y, X).fit()

    # coefficients
    b = model.params[0]   # wealth coefficient
    n = model.params[1]   # risk/loading term

    c = -b / n
    rra_alpha_vector = c * (W ** (1 - h))
    ara_alpha_vector = c / (W ** h)

    results = pd.DataFrame({
        'Year': df.loc[W.index, 'TIME_PERIOD'],
        'optimal_h': h,
        'rra': rra_alpha_vector,
        'ara': ara_alpha_vector
    }).set_index('Year')

    return {
        "alpha": results,
        "c": c,
        "h": h,
        "model_summary": model.summary()
    }

In [ ]:
# for loc in final_df['REF_AREA'].unique():
for loc in final_df['Reference area'].unique():
  # for target in ['claims', 'premiums']:
  results = compute_risk(final_df, loc, 'claims')
  print(f"{loc} {'claims'} {results['alpha']} {results['h']} {results['c']}")

/tmp/ipykernel_553/3392619020.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  b = model.params[0]   # wealth coefficient
/tmp/ipykernel_553/3392619020.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  n = model.params[1]   # risk/loading term


Argentina claims       optimal_h        rra       ara
Year                                
2013        4.2   2.320465  0.000051
2014        4.2   2.509006  0.000056
2015        4.2   2.586283  0.000059
2016        4.2   4.416571  0.000118
2017        4.2   3.737618  0.000095
2018        4.2   7.716258  0.000246
2019        4.2  10.368106  0.000362
2020        4.2  16.207230  0.000651
2021        4.2   7.440974  0.000234
2022        4.2   4.037110  0.000105
2023        4.2   4.235274  0.000112
2024        4.2   4.977647  0.000138 4.200000000000001 1892235347611046.8


/tmp/ipykernel_553/3392619020.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  b = model.params[0]   # wealth coefficient
/tmp/ipykernel_553/3392619020.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  n = model.params[1]   # risk/loading term


Australia claims       optimal_h       rra       ara
Year                               
2013       3.16  2.279844  0.000032
2014       3.16  2.851610  0.000044
2015       3.16  3.520245  0.000060
2016       3.16  4.989914  0.000100
2017       3.16  4.208125  0.000078
2018       3.16  3.984764  0.000072
2019       3.16  4.564941  0.000088
2020       3.16  5.172067  0.000106
2021       3.16  4.111021  0.000076
2022       3.16  4.331993  0.000082
2023       3.16  4.735093  0.000093
2024       3.16  4.690825  0.000092 3.16 69507764809.47563


/tmp/ipykernel_553/3392619020.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  b = model.params[0]   # wealth coefficient
/tmp/ipykernel_553/3392619020.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  n = model.params[1]   # risk/loading term


...

Mexico claims       optimal_h       rra       ara
Year                               
2013       2.17  2.938753  0.000051
2014       2.17  3.065505  0.000055
2015       2.17  3.762065  0.000080
2016       2.17  5.265267  0.000150
2017       2.17  5.774096  0.000178
2018       2.17  5.034173  0.000138
2019       2.17  4.762061  0.000124
2020       2.17  5.989120  0.000190
2021       2.17  4.942419  0.000133
2022       2.17  4.723465  0.000122
2023       2.17  3.852240  0.000084
2024       2.17  3.709792  0.000078 2.17 1098306.753137382


/tmp/ipykernel_553/3392619020.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  b = model.params[0]   # wealth coefficient
/tmp/ipykernel_553/3392619020.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  n = model.params[1]   # risk/loading term


Malaysia claims       optimal_h       rra       ara
Year                               
2013       2.19  3.469319  0.000236
2014       2.19  3.376570  0.000224
2015       2.19  4.037665  0.000311
2016       2.19  4.128346  0.000324
2017       2.19  4.039525  0.000312
2018       2.19  4.762754  0.000422
2019       2.19  5.238184  0.000503
2020       2.19  6.025757  0.000651
2021       2.19  6.175433  0.000681
2022       2.19  6.297929  0.000706
2023       2.19  6.568907  0.000763
2024       2.19  6.568907  0.000763 2.1900000000000004 316437.6162469103
Netherlands claims       optimal_h       rra       ara
Year                               
2013       0.25  1.141943  0.000018
2014       0.25  1.140627  0.000018
2015       0.25  1.009219  0.000019
2016       0.25  1.023025  0.000019
2017       0.25  1.059531  0.000018
2018       0.25  1.121936  0.000018
2019       0.25  1.136659  0.000018
2020       0.25  1.151317  0.000018
2021       0.25  1.165914  0.000018
2022       0.25  1.050561  0

/tmp/ipykernel_553/3392619020.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  b = model.params[0]   # wealth coefficient
/tmp/ipykernel_553/3392619020.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  n = model.params[1]   # risk/loading term


KeyboardInterrupt: 

# Write Results to CSV

In [ ]:
alpha_dfs = {}

# for loc in final_df['REF_AREA'].unique():
for loc in final_df['Reference area'].unique():
    res = compute_risk(final_df, loc, 'claims')
    alpha_dfs[loc] = res['alpha']

results_df = pd.concat(alpha_dfs, ignore_index=False)
# results_df.index.name = 'REF_AREA'
results_df.index.name = 'Reference area'

results_df.to_csv('/content/drive/My Drive/490 project/estimated_risk_aversion(1).csv')

/tmp/ipykernel_553/3392619020.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  b = model.params[0]   # wealth coefficient
/tmp/ipykernel_553/3392619020.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  n = model.params[1]   # risk/loading term
/tmp/ipykernel_553/3392619020.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  b = model.params[0]   # wealth coefficient
/tmp/ipykernel_553/3392619020.py:14: FutureWarning: Series.

In [ ]:
results_df